# 01 — EDA i kontrakt danych (Sprint 1)

Cel notebooka: poznać dane wybranego datasetu i podjąć decyzje, które trafią do kodu:

- **input normalization** — reguły w `normalize_input()` (`src/asi_project/data.py`),
- **training-only filtering** — reguły w `filter_training_rows()` (jeśli są potrzebne),
- **typy cech** — listy `features.numeric` i `features.categorical` w `project.yaml`,
- **metryka główna** — uzasadnienie w karcie projektu w README.

Jak pracować:

- uruchamiajcie notebook od góry (Restart & Run All); po zmianie `project.yaml` uruchomcie
  ponownie komórkę ustawień,
- komórki oznaczone **TODO** uzupełnia zespół; odpowiedzi tekstowe wpisujcie w miejsce
  `*(odpowiedź)*`,
- nie zmieniajcie kontraktu prowadzącej (`src/asi_project/datasets.yaml`).

## 0. Ustawienia

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    encode_target,
    feature_list_problems,
    filter_training_rows,
    is_text_column,
    load_dataset,
    normalize_input,
    target_classes,
    validate,
)

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")

print(f"Dataset: {contract.title} ({contract.dataset_id})")
print(f"Target: {contract.target} | klasa pozytywna: {contract.positive_class}")
print(f"Moment predykcji: {contract.prediction_moment}")

## 1. Wczytanie danych

Pełne dane pobiera `python scripts/download_data.py`. Nazwy kolumn są ujednolicane przy
wczytywaniu (białe znaki), wartości — jeszcze nie.

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
print(df_raw.shape)
df_raw.head()

## 2. Inspekcja: typy, braki, liczba unikalnych wartości

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "braki": df_raw.isna().sum(),
        "unikalne": df_raw.nunique(),
    },
)
overview

**Obserwacje (2–4 zdania):** które kolumny mają nieoczekiwany typ? Gdzie są braki?
Które kolumny liczbowe są w rzeczywistości kategoriami?

Żadna kolumna nie ma nieoczekiwanego typu, wszystkie są liczbowe. W danych nie ma żadnych braków. Kategoriami zapisanymi jako liczby są `Complains`, `Tariff Plan`, `Status`, `Age Group` i `Charge Amount`. Do tego `Age` ma tylko 5 wartości i pokrywa się z `Age Group`, więc też jest raczej kategorią niż prawdziwym wiekiem.

## 3. Kontrakt danych

Kontrakt prowadzącej określa target, klasę pozytywną, ALLOWED FEATURES (jedyne kolumny
dozwolone w X) i EXCLUDED FEATURES wraz z powodem wykluczenia.

In [ ]:
problems = validate(df_raw, contract)
print("Kontrakt danych:", "OK" if not problems else problems)
print("ALLOWED FEATURES:", list(contract.allowed_features))
print("EXCLUDED FEATURES:")
for column, reason in contract.excluded_features.items():
    status = "jest w danych" if column in df_raw.columns else "brak w pliku"
    print(f"- {column} ({status}): {reason}")

## 4. Target i klasa pozytywna

In [ ]:
print("Klasy targetu:", target_classes(df_raw[contract.target]))
y = encode_target(df_raw[contract.target].dropna(), contract.positive_class)
print(f"Udział klasy pozytywnej: {y.mean():.3f}")
ax = y.value_counts().sort_index().plot(kind="bar", title="Target (1 = klasa pozytywna)")
ax.set_xlabel("klasa")
plt.show()

**Metryka główna (2–3 zdania):** którą metrykę (accuracy, precision, recall, F1)
wybieracie jako główną i dlaczego, biorąc pod uwagę udział klasy pozytywnej i koszt błędów
w waszym problemie? To samo uzasadnienie wpiszcie do karty projektu w README.

Jako metrykę główną wybieramy F1. Odchodzących klientów jest tylko ok. 15,7%, więc accuracy nic nam nie powie, bo model, który zawsze mówi „nie odejdzie”, miałby ok. 84% accuracy. Chcemy wyłapać jak najwięcej odchodzących klientów, ale też nie dawać niepotrzebnie ofert tym, którzy i tak by zostali, a F1 bierze pod uwagę obie te rzeczy.

## 5. EDA

**TODO:** co najmniej dwie analizy — jedna dla cechy numerycznej, jedna dla kategorycznej.
Wybierajcie wyłącznie spośród ALLOWED FEATURES.

In [ ]:
# TODO (S1): rozkład wybranej cechy numerycznej w podziale na klasy targetu, np.
# df_raw.groupby(contract.target)["<cecha>"].describe()

# Cecha numeryczna: Seconds of Use (łączny czas rozmów) w podziale na klasy targetu
print(df_raw.groupby(contract.target)["Seconds of Use"].describe().round(1))

ax = df_raw.boxplot(column="Seconds of Use", by=contract.target)
ax.set_title("Seconds of Use a Churn")
ax.set_xlabel("Churn (1 = klient odszedł)")
plt.suptitle("")
plt.show()

In [ ]:
# TODO (S1): udział klasy pozytywnej w kategoriach wybranej cechy kategorycznej, np.
# pd.crosstab(df_raw["<cecha>"], df_raw[contract.target], normalize="index")

# Cecha kategoryczna: Complains (czy klient składał reklamację)
print("Liczba klientów:", df_raw["Complains"].value_counts().to_dict())
pd.crosstab(df_raw["Complains"], df_raw[contract.target], normalize="index").round(3)

**TODO — wnioski z EDA (3–5 zdań):**

Klienci, którzy odeszli, rozmawiają dużo mniej: mediana `Seconds of Use` to ok. 1180 s, a u tych, którzy zostali, ok. 3530 s. `Complains` mocno wiąże się z odejściem, bo wśród klientów z reklamacją odeszło aż 83%, a bez reklamacji tylko ok. 10%. Klientów z reklamacją jest jednak niewielu (241 na 3150). Obie cechy powinny być przydatne dla modelu.

## 6. Input normalization

Input normalization działa na pojedynczym rekordzie, nie używa targetu ani statystyk zbioru
i jest idempotentna. Typowe kandydatki: białe znaki, puste napisy, oczywiste reprezentacje
braków, ujednolicenie zapisu wartości. Uzupełnianie braków medianą **nie jest** input
normalization — wymaga `fit`, więc należy do model preprocessing.

In [ ]:
text_columns = [column for column in df_raw.columns if is_text_column(df_raw[column])]
for column in text_columns:
    values = df_raw[column].dropna().astype(str)
    padded = int((values != values.str.strip()).sum())
    empty = int((values.str.strip() == "").sum())
    top = values.value_counts().head(6).to_dict()
    print(f"{column}: spacje na brzegach={padded}, puste={empty}, najczęstsze={top}")

# col tekstowe. braki i wartosci ujemne
print("Kolumny tekstowe:", text_columns)
print("Braki w danych:", int(df_raw.isna().sum().sum()))
print("Wartości ujemne:", int((df_raw.select_dtypes("number") < 0).sum().sum()))

# sprawdzam czy zera to brak danych
zero_use = df_raw["Seconds of Use"] == 0
print("Klienci z 0 w Seconds of Use:", int(zero_use.sum()))
print(
    "Ci sami mają 0 w Frequency of use:",
    bool((df_raw.loc[zero_use, "Frequency of use"] == 0).all()),
)
print(
    "Ci sami mają 0 w Distinct Called Numbers:",
    bool((df_raw.loc[zero_use, "Distinct Called Numbers"] == 0).all()),
)
print(
    "Call Failure u tych klientów:", df_raw.loc[zero_use, "Call Failure"].value_counts().to_dict()
)
print("Status u tych klientów:", df_raw.loc[zero_use, "Status"].value_counts().to_dict())

**TODO — reguły input normalization:** jakie reguły dopisujecie do `normalize_input()` i dlaczego
są bezpieczne przy predykcji pojedynczego rekordu? Po zmianie funkcji uruchomcie `pytest`.

Kod z szablonu u nas nic nie wypisuje, bo nie ma kolumn tekstowych, więc dopisałem pod nim sprawdzenie braków, wartości ujemnych i zer. Nie dopisujemy nowych reguł, zostaje tylko to, co funkcja robi domyślnie (usuwanie spacji z tekstu i zamiana pustych napisów na brak). W naszych danych nie ma kolumn tekstowych, braków ani ujemnych wartości, więc nie ma czego poprawiać. Sprawdziliśmy też 154 klientów, którzy mają 0 w `Seconds of Use`, `Frequency of use` i `Distinct Called Numbers`. To są ci sami klienci, nie mają też żadnych nieudanych połączeń i większość z nich ma `Status` = 2 (nieaktywny), więc te zera oznaczają po prostu brak korzystania z usług, a nie brak danych. Dlatego zostawiamy je jako 0. Domyślne reguły są bezpieczne, bo patrzą tylko na wartości w jednym rekordzie i nie korzystają z targetu ani z innych rekordów.

In [ ]:
normalized = normalize_input(df_raw)
changed = (normalized.astype(str) != df_raw.astype(str)).sum()
changed[changed > 0]

## 7. Training-only filtering

Training-only filtering działa tylko przy przygotowaniu danych treningowych i może jedynie
usuwać rekordy. Domyślnie `filter_training_rows()` usuwa rekordy bez targetu.

In [ ]:
print("Rekordy bez targetu:", int(df_raw[contract.target].isna().sum()))
print("Dokładne duplikaty wierszy:", int(df_raw.duplicated().sum()))
filtered = filter_training_rows(normalize_input(df_raw), contract)
print(f"Po training-only filtering: {len(filtered)} z {len(df_raw)} rekordów")

**TODO — decyzja o training-only filtering:** czy usuwacie dodatkowo jakieś rekordy
(np. dokładne duplikaty)? Dlaczego tak albo dlaczego nie?

Tak, usuwamy 300 dokładnych duplikatów (zostaje 2850 rekordów). Bez tego ten sam rekord mógłby trafić i do treningu, i do testu, przez co wyniki modelu wyglądałyby lepiej niż są naprawdę. Rekordów bez targetu nie ma.

## 8. Typy cech → `project.yaml`

**TODO:** wpiszcie do `project.yaml` listy `features.numeric` i `features.categorical`
(wyłącznie ALLOWED FEATURES), zapiszcie plik i uruchomcie ponownie komórkę ustawień oraz
komórkę poniżej.

In [ ]:
print("Numeryczne:", list(config.numeric_features))
print("Kategoryczne:", list(config.categorical_features))
print("Problemy z deklaracją:", feature_list_problems(contract, config.model_features) or "brak")
print("Kontrakt z deklaracją:", validate(normalize_input(df_raw), contract, config) or "OK")

# ile klientów odchodzi w każdej grupie wiekowej
df_raw.groupby("Age Group")[contract.target].mean().round(3)

## 9. Podsumowanie decyzji

**TODO — w 3–5 punktach:** decyzje o input normalization, training-only filtering, typach cech
i metryce głównej. Te decyzje będą punktem wyjścia do ADR-1 w Sprincie 2.

- **Input normalization:** nie dopisujemy nowych reguł do `normalize_input()`. W danych nie ma tekstu, braków ani ujemnych wartości, a zera w kolumnach z korzystaniem z usług oznaczają nieaktywnych klientów, więc zostają jako 0.
- **Training-only filtering:** usuwamy 300 dokładnych duplikatów (zostaje 2850 rekordów), żeby ten sam rekord nie trafił do treningu i testu jednocześnie.
- **Cechy numeryczne:** `Call Failure`, `Subscription Length`, `Charge Amount`, `Seconds of Use`, `Frequency of use`, `Frequency of SMS`, `Distinct Called Numbers`. `Charge Amount` traktujemy jako liczbę.
- **Cechy kategoryczne:** `Complains`, `Tariff Plan`, `Status`, `Age Group`. `Age Group` jest kategoryczna, bo odchodzą głównie klienci ze środkowych grup, tak jak pokazałem w komórce wyżej, a najmłodsi i najstarsi prawie wcale. `Age` usunęliśmy, bo to ta sama informacja co `Age Group`.
- **Metryka główna:** F1, bo odchodzących klientów jest tylko ok. 15,7% i liczy się dla nas zarówno recall, jak i precision.